In [1]:
# drift_batch_check_fit.py
import jax
import jax.numpy as jnp
import jax.scipy.linalg as jsp

# ---------- SU(N) utilities ----------

def haar_suN(key, shape, N, dtype=jnp.complex64):
    k1, k2 = jax.random.split(key)
    z = (jax.random.normal(k1, shape + (N, N), dtype=jnp.float32) +
         1j * jax.random.normal(k2, shape + (N, N), dtype=jnp.float32)) / jnp.sqrt(2.0)
    z = z.astype(dtype)
    q, r = jnp.linalg.qr(z)
    phase = jnp.exp(-1j * jnp.angle(jnp.diagonal(r, axis1=-2, axis2=-1)))
    q = q * phase[..., None, :]
    detq = jnp.linalg.det(q)
    q = q / detq[..., None, None] ** (1.0 / N)
    return q

def suN_tangent_gaussian(key, shape, N, dtype=jnp.complex64):
    # anti-Hermitian traceless
    k1, k2 = jax.random.split(key)
    a = (jax.random.normal(k1, shape + (N, N), dtype=jnp.float32) +
         1j * jax.random.normal(k2, shape + (N, N), dtype=jnp.float32)) / jnp.sqrt(2.0)
    a = a.astype(dtype)
    x = a - jnp.conjugate(jnp.swapaxes(a, -1, -2))
    tr = jnp.trace(x, axis1=-2, axis2=-1) / N
    x = x - tr[..., None, None] * jnp.eye(N, dtype=dtype)
    return x

# ---------- lattice ops ----------

def shift4(arr, axis, s):
    return jnp.roll(arr, shift=s, axis=axis)

def plaquette(U, mu, nu):
    # U: [L,L,L,L,4,N,N]
    U_mu = U[..., mu, :, :]
    U_nu = U[..., nu, :, :]
    U_mu_x_nu = shift4(U_mu, axis=nu, s=1)
    U_nu_x_mu = shift4(U_nu, axis=mu, s=1)
    U_mu_dag_x_nu = jnp.conjugate(jnp.swapaxes(U_mu_x_nu, -1, -2))
    U_nu_dag = jnp.conjugate(jnp.swapaxes(U_nu, -1, -2))
    return U_mu @ U_nu_x_mu @ U_mu_dag_x_nu @ U_nu_dag

def z_stack(U):
    # returns z_all: [6,L,L,L,L]
    N = U.shape[-1]
    z_list = []
    for mu in range(4):
        for nu in range(mu + 1, 4):
            Up = plaquette(U, mu, nu)
            tr = jnp.real(jnp.trace(Up, axis1=-2, axis2=-1))
            z = 1.0 - (1.0 / N) * tr
            z_list.append(z)
    return jnp.stack(z_list, axis=0)

def zsum_and_Vbar(U):
    z_all = z_stack(U)
    z_sum = jnp.sum(z_all)
    V = 1.0 + jnp.mean(z_all)
    return z_sum, V

def estimate_LV_one(U, beta, eps, mc_samples, key):
    # stochastic finite-diff estimate of L Vbar (beta may be 0)
    L = U.shape[0]
    N = U.shape[-1]

    def one_dir(k):
        Xi = suN_tangent_gaussian(k, (L, L, L, L, 4), N, dtype=U.dtype)
        exp_p = jsp.expm(eps * Xi)
        exp_m = jsp.expm(-eps * Xi)

        U_p = U @ exp_p
        U_m = U @ exp_m

        zsum_p, V_p = zsum_and_Vbar(U_p)
        _,      V_0 = zsum_and_Vbar(U)
        zsum_m, V_m = zsum_and_Vbar(U_m)

        lap = (V_p + V_m - 2.0 * V_0) / (eps ** 2)

        # drift term (auto-zero if beta==0)
        S_p = beta * zsum_p
        S_m = beta * zsum_m
        dS = (S_p - S_m) / (2.0 * eps)
        dV = (V_p - V_m) / (2.0 * eps)

        return lap - dS * dV

    keys = jax.random.split(key, mc_samples)
    vals = jax.vmap(one_dir)(keys)
    return jnp.mean(vals), jnp.std(vals) / jnp.sqrt(mc_samples)

def sample_batch_stats(key, N, L, K, beta, eps, mc):
    key, kU, kL = jax.random.split(key, 3)
    keysU = jax.random.split(kU, K)
    keysL = jax.random.split(kL, K)

    def one(kU_i, kL_i):
        U = haar_suN(kU_i, (L, L, L, L, 4), N)
        _, v = zsum_and_Vbar(U)
        lv, se = estimate_LV_one(U, beta=beta, eps=eps, mc_samples=mc, key=kL_i)
        return v, lv, se

    v, lv, se = jax.vmap(one)(keysU, keysL)
    return v, lv, se

def fit_lambda_b(V, DeltaV):
    # Fit DeltaV ≈ -lambda * V + b by least squares
    # Solve min || [V, 1] [(-lambda), b]^T - DeltaV ||^2
    X = jnp.stack([V, jnp.ones_like(V)], axis=1)    # [K,2]
    y = DeltaV[:, None]                             # [K,1]
    coef, *_ = jnp.linalg.lstsq(X, y, rcond=None)   # [2,1]
    a = coef[0, 0]  # a ≈ -lambda
    b = coef[1, 0]
    lam = -a
    return lam, b

if __name__ == "__main__":
    # params
    N = 3
    L = 2
    K = 128
    mc = 512
    eps = 5e-3

    key = jax.random.PRNGKey(0)

    # (1) beta=0: fit lambda,b from data
    V0, DeltaV0, se0 = sample_batch_stats(key, N, L, K, beta=0.0, eps=eps, mc=mc)
    lam_hat, b_hat = fit_lambda_b(V0, DeltaV0)

    rhs0 = -lam_hat * V0 + b_hat
    err0 = DeltaV0 - rhs0

    print("=== beta=0 fit (Delta V ≈ -lambda V + b) ===")
    print("lambda_hat =", float(lam_hat))
    print("b_hat      =", float(b_hat))
    print("mean(V)    =", float(jnp.mean(V0)))
    print("mean(err)  =", float(jnp.mean(err0)))
    print("max|err|   =", float(jnp.max(jnp.abs(err0))))
    print("mean SE    =", float(jnp.mean(se0)))

    # (2) beta>0: check Lyapunov inequality using fitted lambda,b
    beta = 6.0
    V, LV, se = sample_batch_stats(key, N, L, K, beta=beta, eps=eps, mc=mc)
    rhs = -lam_hat * V + b_hat
    slack = rhs - LV

    print("\n=== beta>0 drift inequality check (using fitted lambda,b) ===")
    print("mean(V)            =", float(jnp.mean(V)))
    print("mean(L V)          =", float(jnp.mean(LV)))
    print("mean(RHS)          =", float(jnp.mean(rhs)))
    print("min slack (rhs-lv) =", float(jnp.min(slack)))
    print("mean slack         =", float(jnp.mean(slack)))
    print("mean SE(L V)       =", float(jnp.mean(se)))


=== beta=0 fit (Delta V ≈ -lambda V + b) ===
lambda_hat = 21.561948776245117
b_hat      = 43.12388229370117
mean(V)    = 2.0005226135253906
mean(err)  = -1.2785429134964943e-05
max|err|   = 0.0901908278465271
mean SE    = 0.02969277650117874

=== beta>0 drift inequality check (using fitted lambda,b) ===
mean(V)            = 2.0005226135253906
mean(L V)          = -7.094058036804199
mean(RHS)          = -0.011284738779067993
min slack (rhs-lv) = 5.618908405303955
mean slack         = 7.082773208618164
mean SE(L V)       = 0.44095945358276367
